# YOLOX device-resident GPU training @320 → detect → show (COCO128)
Full YOLOX-tiny training device-resident (Thrust device_vector + cuBLAS; Focus + depthwise
grouped conv + decoupled head), trusted host SimOTA loss bridged per image; saves best.pt;
then `yolo detect` + displays the result. **Runtime → GPU (T4)**, Run all.
(YOLOX loss is per-image so it's slower; lower EPOCHS for a quicker demo.)


In [ ]:
!nvidia-smi -L


In [ ]:
%cd /content
!rm -rf yolox_cpp
!git clone -q --branch gpu-device https://github.com/yomei-o/yolox_cpp.git
%cd /content/yolox_cpp
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip && unzip -q -o coco128.zip
!wget -q https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_tiny.pth


### 1. Build (device + cuBLAS) and train — saves last.pt/best.pt each epoch


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -DUSE_CUBLAS -Ipure/third_party pure/dtrain_coco_yolox.cpp -lcublas -o dtrain_coco_yolox
EPOCHS=100  # lower (e.g. 30) for a faster demo
!./dtrain_coco_yolox coco128/images/train2017 320 8 $EPOCHS yolox_tiny


### 2. Build the detector CLI (plain C++/g++) and detect with best.pt


In [ ]:
!g++ -O2 -std=c++17 -Ipure/third_party pure/yolo.cpp -o yolo
!./yolo detect --weights best.pt --source coco128/images/train2017/000000000009.jpg --imgsz 320 --conf 0.25 --out det.png


### 3. Show input + detections


In [ ]:
from IPython.display import Image, display
display(Image('coco128/images/train2017/000000000009.jpg', width=480))
display(Image('det.png', width=480))


---
Other sizes: download that `.pth` (Megvii release) and pass e.g. `... 320 8 100 yolox_s`
(nano/tiny/s bd=1, m bd=2, l bd=3, x bd=4; nano is depthwise). `best.pt` loads in the YOLOX reference.
